In [39]:
vids_input = "/Users/yerik/Desktop/_2_DJ_VIDS/__2025/h"

In [40]:
# -----######-----###### ULTRA VERIFIED RENAME: REAL FPS, RES, DURATION, TIMESTAMP -----######-----######
import subprocess, os
from pathlib import Path
from datetime import datetime, timedelta
import cv2

def _video_0506_pipe_GET_realdata_verifiedrename(folder_path):
    folder = Path(folder_path)
    video_files = sorted(list(folder.rglob("*.mp4")) + list(folder.rglob("*.MP4")))
    if not video_files:
        print("❌ No MP4 files found.")
        return

    for f in video_files:
        print(f"\n📼 Verifying: {f.name}")
        os.system(f'open "{f}"')
        keyword = input("🎤 Enter keyword for this clip (e.g. DJ name, event): ").strip().replace(" ", "_")

        try:
            # --- Get creation time from ffprobe ---
            cmd = [
                "ffprobe", "-v", "error",
                "-select_streams", "v:0",
                "-show_entries", "stream_tags=creation_time",
                "-of", "default=noprint_wrappers=1:nokey=0",
                str(f)
            ]
            output = subprocess.check_output(cmd).decode().splitlines()
            creation_line = next((l for l in output if "TAG:creation_time=" in l), None)
            if not creation_line:
                print(f"❌ No creation_time in metadata: {f.name}")
                continue
            creation_time = creation_line.split("=", 1)[1].strip()
            dt = datetime.fromisoformat(creation_time.replace("Z", "+00:00"))

            # --- OpenCV for actual duration, resolution, fps ---
            cap = cv2.VideoCapture(str(f))
            if not cap.isOpened():
                print(f"❌ Cannot open video for frame analysis: {f.name}")
                continue

            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()

            if width == 0 or height == 0 or fps == 0:
                print(f"❌ Invalid resolution/FPS for: {f.name}")
                continue

            duration_sec = int(frame_count / fps)
            duration_min = round(duration_sec / 60)

            # --- Aspect ratio label ---
            ratio = round(width / height, 2)
            if abs(ratio - 16/9) < 0.05:
                aspect = "16_9"
            elif abs(ratio - 4/3) < 0.05:
                aspect = "4_3"
            else:
                aspect = str(ratio).replace(".", "_")

            # --- Compose filename ---
            ext = f.suffix.lower()
            fps_label = round(fps)
            new_name = (
                f"vid_{dt.strftime('%Y_%m_%d')}_Hr_{dt.strftime('%H_%M_%S')}_"
                f"Yfram_{fps_label}_Yvres_{width}x{height}_Yaspr_{aspect}_Ydura_{duration_min}_MIN_{keyword}{ext}"
            )
            new_path = f.with_name(new_name)

            # --- Collision handling ---
            counter = 1
            while new_path.exists():
                new_name = f"{new_path.stem}_no_{counter}{ext}"
                new_path = f.with_name(new_name)
                counter += 1

            print(f"\n🔁 Verified Rename:\n→ {f.name} ➡️ {new_path.name}")
            input("✅ Press ENTER to confirm, or CTRL+C to skip...")
            f.rename(new_path)
            print(f"✅ Renamed to: {new_path.name}")

        except Exception as e:
            print(f"❌ Error: {e}")


In [41]:
_video_0506_pipe_GET_realdata_verifiedrename(vids_input)


📼 Verifying: vid_2025_07_23_Hr_17_18_32.mp4


🎤 Enter keyword for this clip (e.g. DJ name, event):  s


❌ No creation_time in metadata: vid_2025_07_23_Hr_17_18_32.mp4


In [42]:
# -----######-----###### FULL BATCH W/ CUSTOM CUT OPTION + TQDM -----######-----######
from moviepy.editor import VideoFileClip
from pathlib import Path
from tqdm import tqdm
import random

def _video_0506_storybatch_GET_all9_random_custom(input_folder, duration=60):
    input_folder = Path(input_folder)
    video_paths = [p for p in input_folder.iterdir() if p.suffix.lower() == '.mp4']

    if not video_paths:
        return "❌ No .mp4 files found in input folder."

    for video_path in video_paths:
        clip = VideoFileClip(str(video_path))
        total_duration = clip.duration
        usable_range = total_duration - duration

        if usable_range <= 0:
            print(f"⚠️ Skipping {video_path.name} (too short)")
            continue

        output_dir = input_folder / video_path.stem
        output_dir.mkdir(exist_ok=True)

        base_points = [usable_range * i / 9 for i in range(9)]
        random_offsets = [random.uniform(0, usable_range / 9) for _ in range(9)]
        start_times = [min(bp + ro, usable_range) for bp, ro in zip(base_points, random_offsets)]

        print(f"\n📤 Processing: {video_path.name}")
        for i, start_time in tqdm(enumerate(start_times), total=9, desc=f"⏳ {video_path.stem[:20]}"):
            subclip = clip.subclip(start_time, start_time + duration)
            subclip_resized = subclip.resize(height=1080).crop(x_center=subclip.w / 2, width=608)
            out_path = output_dir / f"{video_path.stem}_story_{i+1:02}.mp4"
            subclip_resized.write_videofile(str(out_path), codec="libx264", audio_codec="aac", verbose=False, logger=None)

        print(f"✅ 9 random clips saved for: {video_path.stem}")

        custom_index = 1
        while True:
            response = 'n'#input(f"✂️  Export custom cut from {video_path.name}? (y/n): ").strip().lower()
            if response != 'y':
                break

            range_str = input("🕐 Enter start-end in minutes (e.g. 1.2 - 3.2): ").strip()
            try:
                start_min, end_min = [float(x.strip()) for x in range_str.split("-")]
                start_sec = start_min * 60
                end_sec = end_min * 60

                if end_sec > total_duration or start_sec >= end_sec:
                    print("❌ Invalid range. Try again.")
                    continue

                subclip = clip.subclip(start_sec, end_sec)
                subclip_resized = subclip.resize(height=1080).crop(x_center=subclip.w / 2, width=608)
                out_path = output_dir / f"__custom_{video_path.stem}_{custom_index}.mp4"
                subclip_resized.write_videofile(str(out_path), codec="libx264", audio_codec="aac", verbose=False, logger=None)
                print(f"✅ Custom clip saved: {out_path.name}")
                custom_index += 1

            except Exception as e:
                print(f"❌ Error parsing input: {e}")
                continue

    return "✅ All videos processed with optional custom cuts."


In [43]:

_video_0506_storybatch_GET_all9_random_custom(
    vids_input,
    duration=45
)


📤 Processing: vid_2025_07_23_Hr_17_18_32.mp4


⏳ vid_2025_07_23_Hr_17:  56%|███████████████████████████▊                      | 5/9 [00:59<00:47, 11.83s/it]


KeyboardInterrupt: 

In [16]:
#### CONTd


In [17]:
# -----######-----###### VIDEO MERGE & AUDIO EXTRACTOR -----######-----###### #
import os
import subprocess
from glob import glob
from tqdm import tqdm

def _vid_2107_halfqualmerge_GET_finalcombo(folder_path):
    """
    Compresses all MP4s in folder to 50% quality, merges them, extracts audio as MP3.
    Keeps original MP4s and adds:
        - combined_video.mp4
        - combined_audio.mp3
    """
    # Step 1: Gather .mp4 files
    mp4_files = sorted(glob(os.path.join(folder_path, "*.mp4")))
    temp_files = []

    print("🎥 Compressing each video to ~50% quality...")
    for i, file in enumerate(tqdm(mp4_files)):
        out_path = os.path.join(folder_path, f"_temp_{i}.mp4")
        temp_files.append(out_path)
        # Shrink resolution and reduce bitrate
        cmd = [
            "ffmpeg", "-i", file,
            "-vf", "scale=iw/2:ih/2",  # reduce resolution
            "-b:v", "1000k",  # set bitrate lower (~1 Mbps)
            "-preset", "fast", "-y",
            out_path
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Step 2: Create concat list file
    concat_txt_path = os.path.join(folder_path, "concat_list.txt")
    with open(concat_txt_path, "w") as f:
        for temp in temp_files:
            f.write(f"file '{temp}'\n")

    # Step 3: Combine videos
    final_video = os.path.join(folder_path, "combined_video.mp4")
    print("🎞️ Combining videos...")
    cmd_concat = [
        "ffmpeg", "-f", "concat", "-safe", "0", "-i", concat_txt_path,
        "-c", "copy", "-y", final_video
    ]
    subprocess.run(cmd_concat, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Step 4: Extract audio
    final_audio = os.path.join(folder_path, "combined_audio.mp3")
    print("🎧 Extracting MP3 audio...")
    cmd_audio = [
        "ffmpeg", "-i", final_video,
        "-q:a", "0", "-map", "a", "-y",
        final_audio
    ]
    subprocess.run(cmd_audio, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Step 5: Cleanup
    print("🧹 Cleaning up temporary files...")
    for temp in temp_files:
        os.remove(temp)
    os.remove(concat_txt_path)

    print(f"✅ Done! Exported:\n🧩 {final_video}\n🎵 {final_audio}")


In [18]:
folder = "/Users/yerik/Desktop/_2_DJ_VIDS/_PUMA_VIDS_July_18"  # ⬅️ Change this to your actual folder
_vid_2107_halfqualmerge_GET_finalcombo(folder)


🎥 Compressing each video to ~50% quality...


100%|██████████████████████████████████████████████████████████████████████████| 3/3 [16:51<00:00, 337.29s/it]


🎞️ Combining videos...
🎧 Extracting MP3 audio...
🧹 Cleaning up temporary files...
✅ Done! Exported:
🧩 /Users/yerik/Desktop/_2_DJ_VIDS/_PUMA_VIDS_July_18/combined_video.mp4
🎵 /Users/yerik/Desktop/_2_DJ_VIDS/_PUMA_VIDS_July_18/combined_audio.mp3
